**Machine Learning Project 5**

Task 1. Downloading data - Splitting data - Preprocessing categorical variables

In [1]:
from google.colab import files
import pandas as pd

# TASK 1. Downloading/splitiing/encoding data
print("TASK 1. Downloading/splitiing/encoding data")
# Downloading data
print("Downloading data")
print("Status: Uploading training.csv")
uploaded = files.upload()
train_filename = list(uploaded.keys())[0]
train_df = pd.read_csv(train_filename)

print("\nStatus: Uploading test.csv")
uploaded = files.upload()
test_filename = list(uploaded.keys())[0]
test_df = pd.read_csv(test_filename)

print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print("\nStatus: Data loaded successfully.")

TASK 1. Downloading/splitiing/encoding data
Status: Uploading training.csv


Saving training.csv to training.csv

Status: Uploading test.csv


Saving test.csv to test.csv

Train shape: (72983, 34)
Test shape: (48707, 33)

Status: Data loaded successfully.


In [2]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q

In [3]:
!pip install lightgbm catboost xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.9 MB/s eta 0:00:00


In [4]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Splitting data

print("Splitting data")
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# List of columns
print("\nList of columns:")
print(train_df.columns)

print("\nTrain set head (checking data examples):")
print(train_df.head())

print("- "*40)
print("\nTarget: 'IsBadBuy'")
print(f"\nTarget distribution:\n{train_df['IsBadBuy'].value_counts(normalize=True)}")

# Converting PurchDate to datetime
train_df['PurchDate'] = pd.to_datetime(train_df['PurchDate'])

# Sorting by date
train_df = train_df.sort_values('PurchDate').reset_index(drop=True)

# Creating time-based split (1/3 each for train, validation, test)
print("- "*40)
print("\nStatus: splitting dataset. 1/3 each for train, validation, test")
# Comment: finding boundary dates, assigning all records on boundary date to earlier set
n = len(train_df)
train_idx = int(n * 1/3)
valid_idx = int(n * 2/3)

# Getting boundary dates (dates at the split points)
train_boundary_date = train_df.iloc[train_idx]['PurchDate']
valid_boundary_date = train_df.iloc[valid_idx]['PurchDate']

# Creating splits
df_train = train_df[train_df['PurchDate'] < train_boundary_date].copy()
df_valid = train_df[(train_df['PurchDate'] >= train_boundary_date) &
                    (train_df['PurchDate'] < valid_boundary_date)].copy()
df_test = train_df[train_df['PurchDate'] >= valid_boundary_date].copy()

print(f"Train dates: {df_train['PurchDate'].min()} to {df_train['PurchDate'].max()}")
print(f"Valid dates: {df_valid['PurchDate'].min()} to {df_valid['PurchDate'].max()}")
print(f"Test dates: {df_test['PurchDate'].min()} to {df_test['PurchDate'].max()}")
print(f"\nSplit sizes. Train: {len(df_train)}. Valid: {len(df_valid)}. Test: {len(df_test)}")
print(f"Total: {len(df_train) + len(df_valid) + len(df_test)} (original: {len(train_df)})")
print(f"\nVerifying no overlap:")
print(f"Train max < Valid min: {df_train['PurchDate'].max() < df_valid['PurchDate'].min()}")
print(f"Valid max < Test min: {df_valid['PurchDate'].max() < df_test['PurchDate'].min()}")


Splitting data

Train shape: (72983, 34)
Test shape: (48707, 33)

List of columns:
Index(['RefId', 'IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge',
       'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size',
       'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice',
       'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice',
       'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice',
       'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice',
       'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost'],
      dtype='object')

Train set head (checking data examples):
   RefId  IsBadBuy  PurchDate Auction  VehYear  VehicleAge   Make  \
0      1         0  12/7/2009   ADESA     2006           3  MAZDA   
1      2         0  12/7/2009   ADESA     2004           5  DODGE   
2 

In [6]:
# Preprocessing categorical variables
print("Preprocessing categorical variables")

def preprocess_data(train, valid, test, use_count_encoding=True):

    # Targets for each data set
    y_train = train['IsBadBuy'].values
    y_valid = valid['IsBadBuy'].values
    y_test = test['IsBadBuy'].values

    # Dropping target and identifiers
    drop_cols = ['IsBadBuy', 'RefId', 'PurchDate']
    X_train = train.drop(columns=drop_cols)
    X_valid = valid.drop(columns=drop_cols)
    X_test = test.drop(columns=drop_cols)

    # Identifying categorical and numerical columns
    categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    print(f"\nCategorical columns: {len(categorical_cols)}")
    print(f"Numerical columns: {len(numerical_cols)}")

    # Handling missing values in numerical columns
    for col in numerical_cols:
        mean_val = X_train[col].mean()
        X_train[col].fillna(mean_val, inplace=True)
        X_valid[col].fillna(mean_val, inplace=True)
        X_test[col].fillna(mean_val, inplace=True)

    # Handling categorical variables
    ## count encoding works like this: "codes" are applied according to the frequenc of a label appearing in the data set
    if use_count_encoding:
        for col in categorical_cols:
            counts = X_train[col].value_counts().to_dict() # counts on training data
            ## Applying to all data sets, using 0 for unseen categories:
            X_train[col] = X_train[col].map(counts).fillna(0)
            X_valid[col] = X_valid[col].map(counts).fillna(0)
            X_test[col] = X_test[col].map(counts).fillna(0)
    else:
        encoders = {}
        for col in categorical_cols:
            le = LabelEncoder() ## label encoding for new categories
            # Fitting on training data
            X_train[col] = X_train[col].fillna('MISSING')
            le.fit(X_train[col])
            encoders[col] = le

            ## Transforming training
            X_train[col] = le.transform(X_train[col])

            # Transforming validation/test with unseen category handling
            X_valid[col] = X_valid[col].fillna('MISSING')
            X_valid[col] = X_valid[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )

            X_test[col] = X_test[col].fillna('MISSING')
            X_test[col] = X_test[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )

    return X_train, X_valid, X_test, y_train, y_valid, y_test


X_train, X_valid, X_test, y_train, y_valid, y_test = preprocess_data(
    df_train, df_valid, df_test, use_count_encoding=True
)

print(f"\nProcessed shapes")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")
print(f"X_test: {X_test.shape}  | y_test: {y_test.shape}")

Preprocessing categorical variables

Categorical columns: 14
Numerical columns: 17

Processed shapes
X_train: (24232, 31) | y_train: (24232,)
X_valid: (24414, 31) | y_valid: (24414,)
X_test: (24337, 31)  | y_test: (24337,)


In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Func to use later
##  Gini coefficient calculation (2*AUC - 1):
def calculate_gini(y_true, y_pred_proba):
    auc = roc_auc_score(y_true, y_pred_proba)
    gini = 2 * auc - 1
    return gini

In [9]:
# Finding the best way to divide dataset into two groups
def _find_best_split(self, X, y):   ## for current node

        best_gini = float('inf') ## set to infinity so any real split will be better
        best_feature = None ## which column/feature to split on
        best_threshold = None ## what value to split at

        n_samples, n_features = X.shape

        ## stop splitting if too few samples (preventing overfit):
        if n_samples < self.min_samples_split:
            return None, None

        # Current gini
        current_gini = self._gini_impurity(y)
        ### Comment: gini impurity: how mixed the current node's classes are

        # Trying each feature:
        for feature_idx in range(n_features):
            ## unique values for this feature:
            thresholds = np.unique(X[:, feature_idx])

            # Limiting number of thresholds to avoid slowdown:
            if len(thresholds) > 10:
                thresholds = np.percentile(X[:, feature_idx], np.linspace(0, 100, 11))

            # Trying each threshold:
            for threshold in thresholds:
                ## spliting data:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                # Calculating weighted gini
                left_gini = self._gini_impurity(y[left_mask])
                right_gini = self._gini_impurity(y[right_mask])

                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                weighted_gini = (n_left * left_gini + n_right * right_gini) / n_samples
                ## updatign: best split:
                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold

In [11]:
# TASK 2. Decision Tree Classifier and Decision Tree Regressor

class Node:

    def __init__(self, depth=0):
        self.depth = depth
        self.feature_idx = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None  ##for leaf nodes
        self.n_samples = 0
        self.gini = None
        self.class_counts = None

    def is_leaf(self):
        return self.value is not None ##returning a bool

    # Computing Gini impurity:
    def compute_gini(self, y):
        self.n_samples = len(y)
        if self.n_samples == 0:
            return 0.0

        _, counts = np.unique(y, return_counts=True)
        self.class_counts = counts
        probabilities = counts / self.n_samples
        gini = 1.0 - np.sum(probabilities ** 2)
        self.gini = gini
        return gini

    # Computing standard deviation for regression:
    def compute_std(self, y):
        self.n_samples = len(y)
        if self.n_samples == 0:
            return 0.0
        return np.std(y)

###########-------------------------


class DecisionTreeClassifier:

    def __init__(self, max_depth=None, min_samples_split=2, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.root = None
        self.n_classes = None

        if random_state is not None:
            np.random.seed(random_state)

    def fit(self, X, y):
        # pandas to to numpy if needed:
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        self.n_classes = len(np.unique(y))
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _find_best_split(self, X, y):   ## for current node
        best_gini = float('inf') ##"any split is better"
        best_feature = None
        best_threshold = None

        n_samples, n_features = X.shape

        if n_samples < self.min_samples_split:
            return None, None

        # Current gini
        current_gini = self._gini_impurity(y)

        # Trying each feature
        for feature_idx in range(n_features):
            # Get unique values for this feature
            thresholds = np.unique(X[:, feature_idx])

            # Limiting number of thresholds to avoid slowdown
            if len(thresholds) > 10:
                thresholds = np.percentile(X[:, feature_idx], np.linspace(0, 100, 11))

            # Trying each threshold
            for threshold in thresholds:
                # Splitting data
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                # Calculating weighted gini
                left_gini = self._gini_impurity(y[left_mask])
                right_gini = self._gini_impurity(y[right_mask])

                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                weighted_gini = (n_left * left_gini + n_right * right_gini) / n_samples

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold

    # Gini impurity
    def _gini_impurity(self, y):

        if len(y) == 0:
            return 0.0

        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        gini = 1.0 - np.sum(probabilities ** 2)
        return gini

    def _build_tree(self, X, y, depth): ## *recursively build

        node = Node(depth=depth)
        node.n_samples = len(y)

        # Checking stopping criteria
        if (self.max_depth is not None and depth >= self.max_depth) or \
           len(np.unique(y)) == 1 or \
           len(y) < self.min_samples_split:
            # Create leaf node
            node.value = self._most_common_label(y)
            node.compute_gini(y)
            return node

        # Finding best split
        feature_idx, threshold = self._find_best_split(X, y)

        if feature_idx is None:
            # Creating leaf node
            node.value = self._most_common_label(y)
            node.compute_gini(y)
            return node

        # Splitting data
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask

        # For split info:
        node.feature_idx = feature_idx
        node.threshold = threshold
        node.compute_gini(y)

        # Recursively building children
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return node

    def _most_common_label(self, y):
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]

    def predict(self, X): ## == predict class labels
        if isinstance(X, pd.DataFrame):
            X = X.values
        return np.array([self._predict_sample(x, self.root) for x in X])

    def predict_proba(self, X): ## == predict class probabilities
        if isinstance(X, pd.DataFrame):
            X = X.values
        probas = np.array([self._predict_proba_sample(x, self.root) for x in X])
        return probas

    def _predict_sample(self, x, node):  ## == predict single sample
        if node.is_leaf():
            return node.value

        if x[node.feature_idx] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)

    def _predict_proba_sample(self, x, node): ## == predict probability for a single sample
        if node.is_leaf():
            # Return probability based on class distribution
            if node.class_counts is None:
                return np.array([0.0, 1.0]) if node.value == 1 else np.array([1.0, 0.0])

            proba = np.zeros(self.n_classes)
            for i, count in enumerate(node.class_counts):
                if i < self.n_classes:
                    proba[i] = count / node.n_samples
            return proba

        if x[node.feature_idx] <= node.threshold:
            return self._predict_proba_sample(x, node.left)
        else:
            return self._predict_proba_sample(x, node.right)


###########-------------------------


class DecisionTreeRegressor:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _find_best_split(self, X, y):   # ==best split for regression
        best_score = float('inf')
        best_feature = None
        best_threshold = None

        n_samples, n_features = X.shape

        if n_samples < self.min_samples_split:
            return None, None

        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])

            # Limiting thresholds
            if len(thresholds) > 10:
                thresholds = np.percentile(X[:, feature_idx], np.linspace(0, 100, 11))

            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                # Calculating MSE for both sides
                left_std = np.std(y[left_mask])
                right_std = np.std(y[right_mask])

                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                weighted_std = (n_left * left_std + n_right * right_std) / n_samples

                if weighted_std < best_score:
                    best_score = weighted_std
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth):  ## recursively
        node = Node(depth=depth)
        node.n_samples = len(y)

        if (self.max_depth is not None and depth >= self.max_depth) or \
           len(y) < self.min_samples_split:
            node.value = np.mean(y)
            return node

        feature_idx, threshold = self._find_best_split(X, y)

        if feature_idx is None:
            node.value = np.mean(y)
            return node

        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask

        node.feature_idx = feature_idx
        node.threshold = threshold

        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return node

    def predict(self, X):

        if isinstance(X, pd.DataFrame):
            X = X.values
        return np.array([self._predict_sample(x, self.root) for x in X])

    def _predict_sample(self, x, node):

        if node.is_leaf():
            return node.value

        if x[node.feature_idx] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)

In [12]:
# Task 2 (continuation, Extra Randomized Tree)
class ExtraTreeClassifier(DecisionTreeClassifier):
    def _find_best_split(self, X, y):  ## split with random thresholds
        best_gini = float('inf')
        best_feature = None
        best_threshold = None

        n_samples, n_features = X.shape

        if n_samples < self.min_samples_split:
            return None, None

        for feature_idx in range(n_features):
            # Getting min and max values
            min_val = X[:, feature_idx].min()
            max_val = X[:, feature_idx].max()

            if min_val == max_val:
                continue

            # Random threshold:
            threshold = np.random.uniform(min_val, max_val)

            # Splitting data
            left_mask = X[:, feature_idx] <= threshold
            right_mask = ~left_mask

            if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                continue

            # Calculating weighted gini
            left_gini = self._gini_impurity(y[left_mask])
            right_gini = self._gini_impurity(y[right_mask])

            n_left = np.sum(left_mask)
            n_right = np.sum(right_mask)
            weighted_gini = (n_left * left_gini + n_right * right_gini) / n_samples

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature_idx
                best_threshold = threshold

        return best_feature, best_threshold

In [13]:
# Tasks 3-4: performance check - later int the notebook

In [24]:
# Task 5. RandomForestClassifier
class RandomForestClassifier:
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt',
                 min_samples_split=2, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []

        if random_state is not None:
            np.random.seed(random_state)

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        self.trees = []
        n_samples, n_features = X.shape

        # Determining max_features
        if self.max_features == 'sqrt':
            max_feat = int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            max_feat = int(np.log2(n_features))
        elif isinstance(self.max_features, int):
            max_feat = self.max_features
        else:
            max_feat = n_features

        print(f"\nStatus: Training Random Forest with {self.n_estimators} trees.")
        for i in range(self.n_estimators):
            if (i + 1) % 10 == 0:
                print(f"Training tree {i + 1}/{self.n_estimators}")

            # Bootstrapping sample
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_boot = X[indices]
            y_boot = y[indices]

            # Random feature subset
            feature_indices = np.random.choice(n_features, max_feat, replace=False)
            X_boot_subset = X_boot[:, feature_indices]

            # Training tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split
            )
            tree.fit(X_boot_subset, y_boot)

            self.trees.append((tree, feature_indices))

        return self

    # Predicting probabilities:
    def predict_proba(self, X):

        if isinstance(X, pd.DataFrame):
            X = X.values

        all_probas = []

        for tree, feature_indices in self.trees:
            X_subset = X[:, feature_indices]
            probas = tree.predict_proba(X_subset) #extracting only the features a tree  was trained on
            all_probas.append(probas)

        # Average predictions
        mean_proba = np.mean(all_probas, axis=0)
        return mean_proba

    # Predicting class labels:
    def predict(self, X):

        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

In [25]:
# Task 6. GBDT classifier, Gradient Boosting Decision Tree classifier

class GBDTClassifier:
    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1,
                 max_features=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.init_pred = None

        if random_state is not None:
            np.random.seed(random_state)

    # Sigmoid function
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        self.trees = []
        n_samples, n_features = X.shape

        # Initializing with log odds
        pos_count = np.sum(y == 1)
        neg_count = np.sum(y == 0)
        self.init_pred = np.log(pos_count / neg_count) if neg_count > 0 else 0.0

        # Current predictions in log-odds space
        F = np.full(n_samples, self.init_pred) ##filling an array of length n_samples with self.init_pred values

        print(f"Status: Training GBDT with {self.n_estimators} trees.")
        for i in range(self.n_estimators):
            if (i + 1) % 20 == 0:
                print(f"Training tree {i + 1}/{self.n_estimators}")

            # Calculating probabilities
            p = self._sigmoid(F)

            # Calculating gradients (negative gradient of binary cross-entropy)
            ## gradient = y - p (derivative of -[y*log(p) + (1-y)*log(1-p)])
            residuals = y - p

            # Determining features to use
            if self.max_features is not None:
                if isinstance(self.max_features, int):
                    n_feat = self.max_features
                elif self.max_features == 'sqrt':
                    n_feat = int(np.sqrt(n_features))
                else:
                    n_feat = n_features

                feature_indices = np.random.choice(n_features, n_feat, replace=False)
                X_subset = X[:, feature_indices]
            else:
                feature_indices = np.arange(n_features)
                X_subset = X

            # Fitting tree to residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X_subset, residuals)

            # Updating predictions
            predictions = tree.predict(X_subset)
            F += self.learning_rate * predictions

            self.trees.append((tree, feature_indices))

        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        n_samples = X.shape[0]
        F = np.full(n_samples, self.init_pred)

        for tree, feature_indices in self.trees:
            X_subset = X[:, feature_indices]
            predictions = tree.predict(X_subset)
            F += self.learning_rate * predictions

        # Converting to probabilities
        p = self._sigmoid(F)

        # Returning probabilities for both classes
        return np.column_stack([1 - p, p])

    def predict(self, X):  #predicting class labels
        probas = self.predict_proba(X)
        return (probas[:, 1] > 0.5).astype(int)

**Tree-Based models implementation**

Models list:
- DecisionTreeClassifier
- DecisionTreeRegressor
- RandomForestClassifier
- GBDTClassifier
- ExtraTreesClassifier

In [78]:
from IPython.display import HTML
#display(HTML("<b>This text is bold</b>"))

In [26]:
# Tasks 2 & 3. Custom Decision Tree Classifier
display(HTML("<b>Tasks 2 & 3: Custom Decision Tree Classifier</b>"))
print("Task 2. Custom Decision Tree Classifier")
print("Task 3 Goal: Gini >= 0.1 on valid")

# Training custom tree with max_depth=7
model = DecisionTreeClassifier(max_depth=7, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred_proba_train = model.predict_proba(X_train)[:, 1]
y_pred_proba_valid = model.predict_proba(X_valid)[:, 1]

# Calculating Gini
gini_train = calculate_gini(y_train, y_pred_proba_train)
gini_valid = calculate_gini(y_valid, y_pred_proba_valid)

print(f"\nCustom Decision Tree results (max_depth=7):")
print(f"Train Gini: {gini_train:.4f}")
print(f"Valid Gini: {gini_valid:.4f}")

if gini_valid >= 0.1:
    print(f"\nStatus: Task 3 PASSED. Validation Gini {gini_valid:.4f} >= 0.1")
else:
    print(f"\nStatus: Task 3 FAILED: Validation Gini {gini_valid:.4f} < 0.1")
    print("Parameters to be adjusted.")

Task 2. Custom Decision Tree Classifier
Task 3 Goal: Gini >= 0.1 on valid

Custom Decision Tree results (max_depth=7):
Train Gini: 0.3946
Valid Gini: 0.2929

Status: Task 3 PASSED. Validation Gini 0.2929 >= 0.1


In [27]:
# Task 4. Comparing with Sklearn
from sklearn.tree import DecisionTreeClassifier as SklearnDT

print("Task 4. Comparing custom tree with sklearn")
# Training sklearn tree
sklearn_tree = SklearnDT(max_depth=7, random_state=42)
sklearn_tree.fit(X_train, y_train)

# Predictions
y_pred_sklearn_train = sklearn_tree.predict_proba(X_train)[:, 1]
y_pred_sklearn_valid = sklearn_tree.predict_proba(X_valid)[:, 1]

# Calculating Gini
gini_sklearn_train = calculate_gini(y_train, y_pred_sklearn_train)
gini_sklearn_valid = calculate_gini(y_valid, y_pred_sklearn_valid)

print(f"\nSklearn Decision Tree results (max_depth=7):")
print(f"Train Gini {gini_sklearn_train:.4f}   |  Valid Gini: {gini_sklearn_valid:.4f}")

print(f"\nRESULTS COMPARISON")
print(f"Custom tree valid Gini  {gini_valid:.4f}   |  Sklearn tree valid Gini {gini_sklearn_valid:.4f}")
print(f"Delta: {gini_sklearn_valid - gini_valid:.4f}")

Task 4. Comparing custom tree with sklearn

Sklearn Decision Tree results (max_depth=7):
Train Gini 0.5340   |  Valid Gini: 0.4370

RESULTS COMPARISON
Custom tree valid Gini  0.2929   |  Sklearn tree valid Gini 0.4370
Delta: 0.1442


Sklearn DecisionTreeClassifier's better performance might be better due to the following reasons:

- Optimized split finding
- More sophisticated pruning and regularization
- Better handling of numerical precision
- Optimized Cython implementation (compliler-friendly, faster, more iterations)
- Better handling of feature thresholds
- More robust Gini calculation with edge cases

In [79]:
# Task 5. Implementing RandomForestClassifier. Target Gini >= 0.15
print("Task 5. Implementing RandomForestClassifier. Target Gini >= 0.15")

# Training random forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    max_features='sqrt', # numer of features to use per split: sqrt of n_features
    random_state=42
)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf_train = rf.predict_proba(X_train)[:, 1]
y_pred_rf_valid = rf.predict_proba(X_valid)[:, 1]

# Calculating Gini
gini_rf_train = calculate_gini(y_train, y_pred_rf_train)
gini_rf_valid = calculate_gini(y_valid, y_pred_rf_valid)

print(f"\nRandom Forest results:")
print(f"Train Gini {gini_rf_train:.4f}   |   Valid Gini {gini_rf_valid:.4f}")

if gini_rf_valid >= 0.15:
    print(f"\nStatus: PASSED. Validation Gini {gini_rf_valid:.4f} >= 0.15")
else:
    print(f"\nStatus: FAILED. Validation Gini {gini_rf_valid:.4f} < 0.15")
    print("Suggestion: n_estimators or max_depth to be increased.")

Task 5. Implementing RandomForestClassifier. Target Gini >= 0.15

Status: Training Random Forest with 100 trees.
Training tree 10/100
Training tree 20/100
Training tree 30/100
Training tree 40/100
Training tree 50/100
Training tree 60/100
Training tree 70/100
Training tree 80/100
Training tree 90/100
Training tree 100/100

Random Forest results:
Train Gini 0.7410   |   Valid Gini 0.4450

Status: PASSED. Validation Gini 0.4450 >= 0.15


In [31]:
# Task 6. Gradient Boosting, GBDT
print("Task 6. Implementing GBDT with binary cross-entropy loss")

# Training GBDT
gbdt = GBDTClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    max_features='sqrt',
    random_state=42
)
gbdt.fit(X_train, y_train)

# Predictions
y_pred_gbdt_train = gbdt.predict_proba(X_train)[:, 1]
y_pred_gbdt_valid = gbdt.predict_proba(X_valid)[:, 1]

# Calculating Gini
gini_gbdt_train = calculate_gini(y_train, y_pred_gbdt_train)
gini_gbdt_valid = calculate_gini(y_valid, y_pred_gbdt_valid)

print(f"\nGBDT RESULTS")
print(f"Train Gini: {gini_gbdt_train:.4f}   |   Valid Gini: {gini_gbdt_valid:.4f}")

print(f"\nComparison with Random Forest:")
print(f"Random Forest Valid {gini_rf_valid:.4f}   |   GBDT Valid {gini_gbdt_valid:.4f}")
print(f"Improvement:         {gini_gbdt_valid - gini_rf_valid:.4f}")

Task 6. Implementing GBDT with binary cross-entropy loss
Status: Training GBDT with 100 trees.
Training tree 20/100
Training tree 40/100
Training tree 60/100
Training tree 80/100
Training tree 100/100

GBDT RESULTS
Train Gini: 0.4979   |   Valid Gini: 0.4616

Comparison with Random Forest:
Random Forest Valid 0.4450   |   GBDT Valid 0.4616
Improvement:         0.0166


In [32]:
#Task 7. LightGBM vs. Catboost vs. XGBoost
# Task 7, LightGBM

import lightgbm as lgb

print("Task 7. LightGBM vs. Catboost vs. XGBoost")
print("Task 7 part 1. LightGBM")

# Creating datasets
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

# Parameters
params_lgb = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'max_depth': -1,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1
}

# Training
model_lgb = lgb.train(
    params_lgb,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
)

# Predictions
y_pred_lgb_train = model_lgb.predict(X_train)
y_pred_lgb_valid = model_lgb.predict(X_valid)

# Gini scores
gini_lgb_train = calculate_gini(y_train, y_pred_lgb_train)
gini_lgb_valid = calculate_gini(y_valid, y_pred_lgb_valid)

print(f"\nLightGBM Results:")
print(f"Train Gini {gini_lgb_train:.4f}  |  Valid Gini {gini_lgb_valid:.4f}")
print(f"Best iteration: {model_lgb.best_iteration}")

print("Notes.")
print(f"\nKey LightGBM Features:")
print("- Histogram-based algorithm (bins continuous features)")
print("- Leaf-wise tree growth (splits leaf with max gain)")
print("- GOSS: Gradient-based One-Side Sampling")
print("- EFB: Exclusive Feature Bundling")
print("- Fastest training among GBDT implementations")

Task 7. LightGBM vs. Catboost vs. XGBoost
Task 7 part 1. LightGBM
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.826959	valid's auc: 0.745106
[100]	train's auc: 0.868152	valid's auc: 0.746947
[150]	train's auc: 0.900822	valid's auc: 0.741566
Early stopping, best iteration is:
[101]	train's auc: 0.868759	valid's auc: 0.747036

LightGBM Results:
Train Gini 0.7375  |  Valid Gini 0.4941
Best iteration: 101
Notes.

Key LightGBM Features:
- Histogram-based algorithm (bins continuous features)
- Leaf-wise tree growth (splits leaf with max gain)
- GOSS: Gradient-based One-Side Sampling
- EFB: Exclusive Feature Bundling
- Fastest training among GBDT implementations


In [34]:
# Task 7, CatBoost
print("Task 7 part 2. CatBoost")

import catboost as cb

# Creating pool objects
train_pool = cb.Pool(X_train, y_train)
valid_pool = cb.Pool(X_valid, y_valid)

# Parameters
params_cb = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'od_type': 'Iter',
    'od_wait': 50,
    'l2_leaf_reg': 3,
    'bagging_temperature': 1,
    'random_strength': 1,
    'verbose': 50
}

# Training
model_cb = cb.CatBoostClassifier(**params_cb)
model_cb.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
    plot=False
)

# Predictions
y_pred_cb_train = model_cb.predict_proba(X_train)[:, 1]
y_pred_cb_valid = model_cb.predict_proba(X_valid)[:, 1]

# Gini scores
gini_cb_train = calculate_gini(y_train, y_pred_cb_train)
gini_cb_valid = calculate_gini(y_valid, y_pred_cb_valid)

print(f"\nCatBoost results")
print(f"Train Gini: {gini_cb_train:.4f}  |  Valid Gini: {gini_cb_valid:.4f}")
print(f"Best iteration: {model_cb.get_best_iteration()}")

print("\nNotes.")
print(f"Key CatBoost Features:")
print("- Ordered boosting (prevents prediction shift)")
print("- Symmetric (oblivious) trees")
print("- Advanced categorical encoding:")
print("  * Target Statistics with proper validation")
print("  * Automatic categorical combinations")
print("- Most robust to overfitting")

Task 7 part 2. CatBoost
0:	test: 0.6592873	best: 0.6592873 (0)	total: 17.4ms	remaining: 17.4s
50:	test: 0.7420143	best: 0.7420143 (50)	total: 871ms	remaining: 16.2s
100:	test: 0.7446545	best: 0.7451716 (82)	total: 1.7s	remaining: 15.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7451716112
bestIteration = 82

Shrink model to first 83 iterations.

CatBoost results
Train Gini: 0.5583  |  Valid Gini: 0.4903
Best iteration: 82

Notes.
Key CatBoost Features:
- Ordered boosting (prevents prediction shift)
- Symmetric (oblivious) trees
- Advanced categorical encoding:
  * Target Statistics with proper validation
  * Automatic categorical combinations
- Most robust to overfitting


In [35]:
# Task 7, XGBoost
print("Task 7 part 3. XGBoost")

import xgboost as xgb

# Creating DMatrix objects
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

# Standard GBDT parameters
params_xgb = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 1,
    'seed': 42
}

# Training
model_xgb = xgb.train(
    params_xgb,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=50
)

# Predictions
y_pred_xgb_train = model_xgb.predict(dtrain)
y_pred_xgb_valid = model_xgb.predict(dvalid)

# Gini scores
gini_xgb_train = calculate_gini(y_train, y_pred_xgb_train)
gini_xgb_valid = calculate_gini(y_valid, y_pred_xgb_valid)

print(f"\nXGBoost Standard results")
print(f"Train Gini {gini_xgb_train:.4f}  |  Valid Gini {gini_xgb_valid:.4f}")
print(f"Best iteration: {model_xgb.best_iteration}")

print("\nNotes.")
print(f"\nKey XGBoost Features:")
print("- Level-wise tree growth (all nodes at same level split)")
print("- L1/L2 regularization")
print("- Sparsity awareness")

Task 7 part 3. XGBoost
[0]	train-auc:0.75822	valid-auc:0.71215
[50]	train-auc:0.82053	valid-auc:0.74538
[100]	train-auc:0.85547	valid-auc:0.74554
[131]	train-auc:0.87635	valid-auc:0.74332

XGBoost Standard results
Train Gini 0.7527  |  Valid Gini 0.4866
Best iteration: 81

Notes.

Key XGBoost Features:
- Level-wise tree growth (all nodes at same level split)
- L1/L2 regularization
- Sparsity awareness


In [54]:
print("GBDT comparison")

results = {
    'LightGBM': (gini_lgb_train, gini_lgb_valid),
    'CatBoost': (gini_cb_train, gini_cb_valid),
    'XGBoost': (gini_xgb_train, gini_xgb_valid),
}

print(f"\n{'Model':<15} {'Train Gini':>12} {'Valid Gini':>12} {'Overfit Gap':>12}")
print("-"*53)

for name, (train_gini, valid_gini) in results.items():
    gap = train_gini - valid_gini
    print(f"{name:<15} {train_gini:>12.4f} {valid_gini:>12.4f} {gap:>12.4f}")

# Find best model
best_model_name = max(results.keys(), key=lambda k: results[k][1])
best_valid_gini = results[best_model_name][1]

print(f"\nBEST MODEL: {best_model_name}")
print(f"Validation Gini: {best_valid_gini:.4f}")

GBDT comparison

Model             Train Gini   Valid Gini  Overfit Gap
-----------------------------------------------------
LightGBM              0.7375       0.4941       0.2434
CatBoost              0.5583       0.4903       0.0679
XGBoost               0.7527       0.4866       0.2660

BEST MODEL: LightGBM
Validation Gini: 0.4941


In [59]:
# Task 8. Best model performance analysis
print("Task 8. Best model performance analysis")

# Test predictions for the best model (best_model_name == 'LightGBM')
y_pred_test = model_lgb.predict(X_test)
model_best = model_lgb

# Get train and valid predictions
y_pred_train_final = model_lgb.predict(X_train)
y_pred_valid_final = model_lgb.predict(X_valid)

# Calculate Gini scores
gini_train_final = calculate_gini(y_train, y_pred_train_final)
gini_valid_final = calculate_gini(y_valid, y_pred_valid_final)
gini_test_final = calculate_gini(y_test, y_pred_test)

print(f"\nBest Model: {best_model_name}")
print(f"\n{'Dataset':<15} {'Gini Score':>12}")
print("-"*55)
print(f"{'Training':<15} {gini_train_final:>12.4f}")
print(f"{'Validation':<15} {gini_valid_final:>12.4f}")
print(f"{'Test':<15} {gini_test_final:>12.4f}")

# Metric drop
valid_test_drop = gini_valid_final - gini_test_final
train_test_gap = gini_train_final - gini_test_final
train_valid_gap = gini_train_final - gini_valid_final

print("\nOVERFITTING ANALYSIS")
print(f"Valid-Test drop  {valid_test_drop:+.4f}")
print(f"Train-Test gap   {train_test_gap:.4f}")
print(f"Train-Valid gap  {train_valid_gap:.4f}")

Task 8. Best model performance analysis

Best Model: LightGBM

Dataset           Gini Score
-------------------------------------------------------
Training              0.7375
Validation            0.4941
Test                  0.5054

OVERFITTING ANALYSIS
Valid-Test drop  -0.0113
Train-Test gap   0.2321
Train-Valid gap  0.2434


In [64]:
# Task 9 (bonus). Implementing the ExtraTreesClassifier

class ExtraTreesClassifier:
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt',
                 min_samples_split=2, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []

        if random_state is not None:
            np.random.seed(random_state)

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        self.trees = []
        n_samples, n_features = X.shape

        # Determining max_features
        if self.max_features == 'sqrt':
            max_feat = int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            max_feat = int(np.log2(n_features))
        elif isinstance(self.max_features, int):
            max_feat = self.max_features
        else:
            max_feat = n_features

        print(f"\nStatus: Training Extra Trees with {self.n_estimators} trees.")
        for i in range(self.n_estimators):
            if (i + 1) % 10 == 0:
                print(f"Training tree {i + 1}/{self.n_estimators}")

            # Using full dataset
            ## random feature subset:
            feature_indices = np.random.choice(n_features, max_feat, replace=False)
            X_subset = X[:, feature_indices]

            # Training extra tree
            tree = ExtraTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                random_state=self.random_state + i if self.random_state else None
            )
            tree.fit(X_subset, y)

            self.trees.append((tree, feature_indices))

        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        all_probas = []

        for tree, feature_indices in self.trees:
            X_subset = X[:, feature_indices]
            probas = tree.predict_proba(X_subset)
            all_probas.append(probas)

        mean_proba = np.mean(all_probas, axis=0)
        return mean_proba

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

In [66]:
# Task 9 (continuation)
print("Task 9 (bonus). Implementing the ExtraTreesClassifier")

# Training Extra Trees
extra_trees = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=10,
    max_features='sqrt',
    random_state=42
)
extra_trees.fit(X_train, y_train)

# Predictions
y_pred_et_train = extra_trees.predict_proba(X_train)[:, 1]
y_pred_et_valid = extra_trees.predict_proba(X_valid)[:, 1]

# Calculating Gini
gini_et_train = calculate_gini(y_train, y_pred_et_train)
gini_et_valid = calculate_gini(y_valid, y_pred_et_valid)

print(f"\nExtra Trees Results:")
print(f"Train Gini: {gini_et_train:.4f}")
print(f"Valid Gini: {gini_et_valid:.4f}")

if gini_et_valid >= 0.12:
    print(f"\nPASSED: Validation Gini {gini_et_valid:.4f} >= 0.12")
else:
    print(f"\nFAILED: Validation Gini {gini_et_valid:.4f} < 0.12")

Task 9 (bonus). Implementing the ExtraTreesClassifier

Status: Training Extra Trees with 100 trees.
Training tree 10/100
Training tree 20/100
Training tree 30/100
Training tree 40/100
Training tree 50/100
Training tree 60/100
Training tree 70/100
Training tree 80/100
Training tree 90/100
Training tree 100/100

Extra Trees Results:
Train Gini: 0.6280
Valid Gini: 0.4689

PASSED: Validation Gini 0.4689 >= 0.12


In [76]:
# Summary of custom models
print("Summary of custom models")

results_custom = {
    'Decision Tree': gini_valid,
    'Random Forest': gini_rf_valid,
    'GBDT': gini_gbdt_valid,
    'Extra Trees': gini_et_valid
}

print(f"\n{'Model':<20} {'Valid Gini':>12}")
print("-"*37)
for name, gini in results_custom.items():
    print(f"{name:<20} {gini:>12.4f}")

best_custom = max(results_custom, key=results_custom.get)
print(f"\nBest Custom Model: {best_custom} (Gini: {results_custom[best_custom]:.4f})")

Summary of custom models

Model                  Valid Gini
-------------------------------------
Decision Tree              0.2929
Random Forest              0.4450
GBDT                       0.4616
Extra Trees                0.4689

Best Custom Model: Extra Trees (Gini: 0.4689)


In [77]:
print("COMPLETE RESULTS SUMMARY")

all_results = {
    'Custom Tree': gini_valid,
    'Custom RF': gini_rf_valid,
    'Custom GBDT': gini_gbdt_valid,
    'Custom ExtraTrees': gini_et_valid,
    'LightGBM': gini_lgb_valid,
    'CatBoost': gini_cb_valid,
    'XGBoost': gini_xgb_valid,
}

print(f"{'Model':<20} {'Valid Gini':>12}")
print("-"*50)

for name, gini in all_results.items():
    print(f"{name:<20} {gini:>12.4f}")

best_overall = max(all_results, key=all_results.get)
print(f"\nBest Overall Model: {best_overall}")
print(f"Validation Gini: {all_results[best_overall]:.4f}")
print(f"Test Gini: {gini_test_final:.4f}")


COMPLETE RESULTS SUMMARY
Model                  Valid Gini
--------------------------------------------------
Custom Tree                0.2929
Custom RF                  0.4450
Custom GBDT                0.4616
Custom ExtraTrees          0.4689
LightGBM                   0.4941
CatBoost                   0.4903
XGBoost                    0.4866

Best Overall Model: LightGBM
Validation Gini: 0.4941
Test Gini: 0.5054
